In [1]:
# connect to the sql server from the notebook
import pandas as pd
import numpy as np
import pyodbc
import warnings

In [2]:
# suppress pandas SQL connection warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [3]:
# connect to the database
connector = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=LAPPY\\SQLEXPRESS;'
    'DATABASE=DunnhumbyDB;'
    'Trusted_Connection=yes;'
)
print("Successfully connected to SQL Server!")

Successfully connected to SQL Server!


In [4]:
# extract full time boundaries across all Silver tables
query_boundaries = """
SELECT 'silver.transaction_data (day)' AS source_table, MIN(day) AS min_value, MAX(day) AS max_value, COUNT(DISTINCT day) AS distinct_values
FROM silver.transaction_data
UNION ALL
SELECT 'silver.campaign_desc (start_day & end_day)', MIN(start_day), MAX(end_day), NULL
FROM silver.campaign_desc
UNION ALL
SELECT 'silver.coupon_redempt (day)', MIN(day), MAX(day), COUNT(DISTINCT day)
FROM silver.coupon_redempt
UNION ALL
SELECT 'silver.causal_data (week_no)', MIN(week_no), MAX(week_no), COUNT(DISTINCT week_no)
FROM silver.causal_data
UNION ALL
SELECT 'silver.transaction_data (week_no)', MIN(week_no), MAX(week_no), COUNT(DISTINCT week_no)
FROM silver.transaction_data;
"""

df_boundaries = pd.read_sql(query_boundaries, connector)
print(df_boundaries)

                                 source_table  min_value  max_value  \
0               silver.transaction_data (day)          1        711   
1  silver.campaign_desc (start_day & end_day)        224        719   
2                 silver.coupon_redempt (day)        225        704   
3                silver.causal_data (week_no)          9        101   
4           silver.transaction_data (week_no)          1        102   

   distinct_values  
0            711.0  
1              NaN  
2            328.0  
3             93.0  
4            102.0  


In [5]:
# Summary:
# - Transactions span Days 1–711 (711 distinct days) and Weeks 1–102.
# - Campaigns are active from Day 224 to Day 719.
# - Coupon redemptions occur on 328 distinct days between Days 225 and 704.
# - Causal data is available only for Weeks 9–101 (93 distinct weeks).
# - The Gold date dimension should cover the full range of Days 1–719.
# Calendar: Day 1 = 2020-01-01, Day 719 = 2021-12-19

In [6]:
# extract & inspect product demographics & commodities
query_product = """
SELECT product_id, department, commodity_desc, sub_commodity_desc, is_unknown_product 
FROM silver.product
"""
df_product = pd.read_sql(query_product, connector)

print(f"Total Products: {len(df_product)}")
print(f"Flagged Unknown Products: {df_product['is_unknown_product'].sum()}")
print(f"Missing Commodity Descriptions: {df_product['commodity_desc'].isnull().sum()}")
print(f"Unique Departments: {df_product['department'].nunique()}")
print(f"Unique Commodities: {df_product['commodity_desc'].nunique()}")
df_product.head()

Total Products: 92353
Flagged Unknown Products: 67
Missing Commodity Descriptions: 0
Unique Departments: 44
Unique Commodities: 308


,product_id,department,commodity_desc,sub_commodity_desc,is_unknown_product
0,25671,GROCERY,FRZN ICE,ICE - CRUSHED/CUBED,False
1,26081,MISC. TRANS.,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,False
2,26093,PASTRY,BREAD,BREAD:ITALIAN/FRENCH,False
3,26190,GROCERY,FRUIT - SHELF STABLE,APPLE SAUCE,False
4,26355,GROCERY,COOKIES/CONES,SPECIALTY COOKIES,False


In [7]:
# extract campaign exposure windows
query_campaign_windows = """SELECT ct.household_key, ct.campaign, cd.description, cd.start_day, cd.end_day
                            FROM silver.campaign_table ct
                            JOIN silver.campaign_desc cd ON ct.campaign = cd.campaign"""

df_exposure = pd.read_sql(query_campaign_windows, connector)
print(f"Total Household-Campaign Exposure Records: {len(df_exposure)}")
print(
    f"Unique Households with Active Campaigns:"
    f" {df_exposure['household_key'].nunique()}"
)
df_exposure.head()

Total Household-Campaign Exposure Records: 7208
Unique Households with Active Campaigns: 1584


,household_key,campaign,description,start_day,end_day
0,1,8,TypeA,412,460
1,1,12,TypeB,477,509
2,1,13,TypeA,504,551
3,1,18,TypeA,587,642
4,1,20,TypeC,615,685


In [8]:
# pull a representative sample of household transactions
query_sample_transactions = """SELECT t.*
                               FROM silver.transaction_data t
                               WHERE t.household_key IN (SELECT DISTINCT TOP 50 household_key FROM silver.campaign_table)"""

df_trans_sample = pd.read_sql(query_sample_transactions, connector)
print(f"Total Transactions in Sample: {len(df_trans_sample)}")
print(
    f"Distinct Households in Sample: {df_trans_sample['household_key'].nunique()}"
)
print(
    f"Transaction Days Covered: Min Day {df_trans_sample['day'].min()} to Max"
    f" Day {df_trans_sample['day'].max()}"
)
df_trans_sample.head()

Total Transactions in Sample: 69095
Distinct Households in Sample: 50
Transaction Days Covered: Min Day 4 to Max Day 711


,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc,dwh_create_date
0,68,33251687227,426,1085604,1,1.00,32004,-0.29,2009,62,0.0,0.0,2026-08-08 16:16:24.536666
1,68,33251687227,426,13842224,1,1.00,32004,-0.29,2009,62,0.0,0.0,2026-08-08 16:16:24.536666
2,68,33251687356,426,929652,1,0.89,32004,0.00,2057,62,0.0,0.0,2026-08-08 16:16:24.536666
3,20,33251725168,426,949836,2,1.58,412,-0.80,728,62,0.0,0.0,2026-08-08 16:16:24.536666
4,20,33251725168,426,1075368,1,3.84,412,0.00,728,62,0.0,0.0,2026-08-08 16:16:24.536666


In [9]:
# map transactions to calendar dates & attach product details
ANCHOR_DATE = pd.to_datetime("2020-01-01")

# 1. Map transaction day number to actual calendar date
df_trans_sample["trans_date"] = df_trans_sample["day"].apply(
    lambda d: (ANCHOR_DATE + pd.Timedelta(days=int(d) - 1)).strftime("%Y-%m-%d"))

# 2. Attach Product details
df_trans_sample = df_trans_sample.merge(
    df_product[["product_id", "department", "commodity_desc"]],
    on="product_id",
    how="left",)

print(f"Sample Transactions: {len(df_trans_sample)}")
print(f"Missing Commodities: {df_trans_sample['commodity_desc'].isnull().sum()}")
df_trans_sample[["household_key", "day", "trans_date", "department", "commodity_desc", "sales_value"]].head()

Sample Transactions: 69095
Missing Commodities: 0


,household_key,day,trans_date,department,commodity_desc,sales_value
0,68,426,2021-03-01,GROCERY,SOFT DRINKS,1.00
1,68,426,2021-03-01,GROCERY,SOFT DRINKS,1.00
2,68,426,2021-03-01,DRUG GM,STATIONERY & SCHOOL SUPPLIES,0.89
3,20,426,2021-03-01,GROCERY,FROZEN PIE/DESSERTS,1.58
4,20,426,2021-03-01,DRUG GM,CIGARETTES,3.84


In [10]:
# calculate household campaign exposure windows
# Create exposure dictionary per household for fast execution
exposure_dict = {
    hh: df_exposure[df_exposure["household_key"] == hh][["start_day", "end_day", "campaign"]]
    for hh in df_trans_sample["household_key"].unique()
}

def is_campaign_active(hh_key, day_num):
    if hh_key not in exposure_dict or exposure_dict[hh_key].empty:
        return False
    df_hh = exposure_dict[hh_key]
    active = df_hh[(df_hh["start_day"] <= day_num) & (df_hh["end_day"] >= day_num)]
    return not active.empty

# Apply active campaign check
df_trans_sample["is_during_campaign"] = df_trans_sample.apply(
    lambda row: is_campaign_active(row["household_key"], row["day"]), axis=1
)

print("Transaction Counts by Campaign Active Status:")
print(df_trans_sample["is_during_campaign"].value_counts())

Transaction Counts by Campaign Active Status:
is_during_campaign
False    44198
True     24897
Name: count, dtype: int64


In [11]:
# compute non-campaign household baseline spend rates
# Filter baseline transactions
df_baseline = df_trans_sample[df_trans_sample["is_during_campaign"] == False]

# Aggregate baseline by household & commodity
hh_category_baseline = (
    df_baseline.groupby(["household_key", "commodity_desc"])
    .agg(
        baseline_sales=("sales_value", "sum"),
        baseline_active_days=("day", "nunique"),
    )
    .reset_index()
)

# Normalize baseline spend per active day
hh_category_baseline["baseline_spend_per_day"] = (
    hh_category_baseline["baseline_sales"] / hh_category_baseline["baseline_active_days"]
)

print(f"Total Household-Commodity Baseline Pairs: {len(hh_category_baseline)}")
hh_category_baseline.sort_values("baseline_sales", ascending=False).head()

Total Household-Commodity Baseline Pairs: 5890


,household_key,commodity_desc,baseline_sales,baseline_active_days,baseline_spend_per_day
898,13,COUPON/MISC ITEMS,2350.05,53,44.340566
4736,58,COUPON/MISC ITEMS,1808.00,48,37.666667
5556,71,CIGARETTES,1563.89,45,34.753111
3116,40,COUPON/MISC ITEMS,1522.47,59,25.804576
2829,37,SOFT DRINKS,1380.78,21,65.751429


In [12]:
# compute campaign active spend rates
# Filter campaign transactions
df_campaign = df_trans_sample[df_trans_sample["is_during_campaign"] == True]

# Aggregate campaign by household & commodity
hh_category_campaign = (
    df_campaign.groupby(["household_key", "commodity_desc"])
    .agg(
        campaign_sales=("sales_value", "sum"),
        campaign_active_days=("day", "nunique"),
    )
    .reset_index()
)

# Normalize campaign spend per active day
hh_category_campaign["campaign_spend_per_day"] = (
    hh_category_campaign["campaign_sales"] / hh_category_campaign["campaign_active_days"]
)

print(f"Total Household-Commodity Campaign Pairs: {len(hh_category_campaign)}")
hh_category_campaign.sort_values("campaign_sales", ascending=False).head()

Total Household-Commodity Campaign Pairs: 4149


,household_key,commodity_desc,campaign_sales,campaign_active_days,campaign_spend_per_day
579,13,COUPON/MISC ITEMS,2135.89,42,50.854524
2139,40,COUPON/MISC ITEMS,1486.69,52,28.590192
1524,27,BEERS/ALES,1184.46,183,6.472459
3192,58,COUPON/MISC ITEMS,1028.19,25,41.127600
1535,27,CIGARETTES,821.06,127,6.465039


In [13]:
# compute incremental lift & apply robustness filters
# Merge Baseline and Campaign metrics
df_lift_proto = hh_category_baseline.merge(
    hh_category_campaign,
    on=["household_key", "commodity_desc"],
    how="outer",
).fillna(0)

# Calculate Incremental Lift Per Day
df_lift_proto["daily_incremental_lift"] = (
    df_lift_proto["campaign_spend_per_day"] - df_lift_proto["baseline_spend_per_day"]
)

# Filter for reliable pairs (at least 3 active days in both windows)
df_reliable_lift = df_lift_proto[
    (df_lift_proto["baseline_active_days"] >= 3) & 
    (df_lift_proto["campaign_active_days"] >= 3)
]

print(f"Total Evaluated Pairs: {len(df_lift_proto)}")
print(f"Reliable Evaluated Pairs (>=3 active days): {len(df_reliable_lift)}")

print("\n--- Top 5 Highest Performing Categories by Daily Incremental Lift ---")
print(
    df_reliable_lift.sort_values("daily_incremental_lift", ascending=False)[
        ["household_key", "commodity_desc", "baseline_spend_per_day", "campaign_spend_per_day", "daily_incremental_lift"]
    ].head().to_string(index=False)
)

Total Evaluated Pairs: 6563
Reliable Evaluated Pairs (>=3 active days): 1622

--- Top 5 Highest Performing Categories by Daily Incremental Lift ---
 household_key        commodity_desc  baseline_spend_per_day  campaign_spend_per_day  daily_incremental_lift
            52     COUPON/MISC ITEMS               33.087143               54.088000               21.000857
            58 DIAPERS & DISPOSABLES               10.620000               25.656667               15.036667
            40                 CAKES                9.430000               21.160000               11.730000
            67     COUPON/MISC ITEMS               18.506429               28.025714                9.519286
            67  MAKEUP AND TREATMENT                5.631667               14.700000                9.068333


In [14]:
# Summary:
# -COUPON/MISC ITEMS identified as a data issue → catch-all category causing inflated lift rankings; should be 
#  excluded/flagged.
# -Robustness filter applied successfully → reduced 6,638 household-category pairs to 1,679 reliable pairs with 
#  sufficient active days.
# -Top genuine incremental drivers (after removing misc items):
#  Diapers & Disposables → +$15.04/day lift
#  Cakes → +$11.73/day lift

In [15]:
# extract causal promotion master data
query_causal = """
SELECT product_id, store_id, week_no, display, mailer, is_unknown_display_code, is_unknown_mailer_code 
FROM silver.causal_data
"""
df_causal = pd.read_sql(query_causal, connector)

print(f"Total Causal Promotion Records: {len(df_causal):,}")
print(f"Unique Featured Products: {df_causal['product_id'].nunique():,}")
print(f"Unique Stores with Promotions: {df_causal['store_id'].nunique():,}")
print(f"Display Flag Counts:\n{df_causal['display'].value_counts()}")
print(f"Mailer Flag Counts:\n{df_causal['mailer'].value_counts()}")
print(f"Unknown Display Codes: {df_causal['is_unknown_display_code'].sum()}")
print(f"Unknown Mailer Codes: {df_causal['is_unknown_mailer_code'].sum()}")
df_causal.head()

Total Causal Promotion Records: 36,786,524
Unique Featured Products: 68,377
Unique Stores with Promotions: 115
Display Flag Counts:
display
Not on Display                21038745
Secondary Location Display     2699467
Rear End Cap                   2575289
In-Aisle                       2362118
Front End Cap                  2073738
Side-Aisle End Cap             1816021
Store Rear                     1812840
Store Front                    1102141
In-Shelf                        713180
Mid-Aisle End Cap               592985
Name: count, dtype: int64
Mailer Flag Counts:
mailer
Interior page feature                    17106789
Not on ad                                11534183
Front page feature                        4467453
Wrap front feature                        1560395
Back page feature                         1077549
Wrap interior coupon                       306924
Wrap back feature                          301327
Interior page line item                    291059
Free on interior 

,product_id,store_id,week_no,display,mailer,is_unknown_display_code,is_unknown_mailer_code
0,2629797,438,97,Not on Display,Front page feature,False,False
1,2629797,438,100,Not on Display,Interior page feature,False,False
2,2629797,438,101,Not on Display,Front page feature,False,False
3,2629797,439,12,Not on Display,Interior page feature,False,False
4,2629797,439,14,Not on Display,Interior page feature,False,False


In [16]:
# aggregate sample transactions at Weekly + Store + Product level
df_weekly_store_sales = (
    df_trans_sample.groupby(["store_id", "product_id", "week_no"]).agg(
        weekly_sales=("sales_value", "sum"),
        weekly_units=("quantity", "sum"),
        active_trans_days=("day", "nunique"),).reset_index())

print(f"Weekly Store-Product Sales Records: {len(df_weekly_store_sales):,}")
df_weekly_store_sales.head()

Weekly Store-Product Sales Records: 66,970


,store_id,product_id,week_no,weekly_sales,weekly_units,active_trans_days
0,145,616830,89,5.00,2,1
1,145,762311,89,2.50,1,1
2,145,797263,89,1.50,1,1
3,195,480014,73,15.33,5381,1
4,289,821976,43,2.00,1,1


In [17]:
# merge weekly sales with causal promotion flags
df_promo_sales = df_weekly_store_sales.merge(
    df_causal, on=["store_id", "product_id", "week_no"], how="left")

# Fill unfeatured weeks with text defaults
df_promo_sales["display"] = df_promo_sales["display"].fillna("Not on Display").astype(str)
df_promo_sales["mailer"] = df_promo_sales["mailer"].fillna("Not on ad").astype(str)

# Create a combined Causal Status Flag matching new Silver text values
df_promo_sales["is_promoted"] = (
    (df_promo_sales["display"] != "Not on Display") | 
    (df_promo_sales["mailer"] != "Not on ad"))

print("Weekly Sales Records by Promotion Status:")
print(df_promo_sales["is_promoted"].value_counts())
df_promo_sales.head()

Weekly Sales Records by Promotion Status:
is_promoted
False    53036
True     13948
Name: count, dtype: int64


,store_id,product_id,week_no,weekly_sales,weekly_units,active_trans_days,display,mailer,is_unknown_display_code,is_unknown_mailer_code,is_promoted
0,145,616830,89,5.00,2,1,Not on Display,Not on ad,NaN,NaN,False
1,145,762311,89,2.50,1,1,Not on Display,Not on ad,NaN,NaN,False
2,145,797263,89,1.50,1,1,Not on Display,Not on ad,NaN,NaN,False
3,195,480014,73,15.33,5381,1,Not on Display,Not on ad,NaN,NaN,False
4,289,821976,43,2.00,1,1,Front End Cap,Not on ad,False,False,True


In [ ]:
# compute baseline vs. promoted weekly sales per store & commodity
# Attach Product Category
df_promo_sales = df_promo_sales.merge(
    df_product[["product_id", "commodity_desc"]], on="product_id", how="left")

# Split into Non-Promoted (Baseline) and Promoted Data
baseline_store_comm = (
    df_promo_sales[df_promo_sales["is_promoted"] == False]
    .groupby(["store_id", "commodity_desc"]).agg(
        baseline_weekly_sales=("weekly_sales", "mean"),
        baseline_weeks_count=("week_no", "nunique"),).reset_index())

promoted_store_comm = (
    df_promo_sales[df_promo_sales["is_promoted"] == True]
    .groupby(["store_id", "commodity_desc"]).agg(
        promoted_weekly_sales=("weekly_sales", "mean"),
        promoted_weeks_count=("week_no", "nunique"),).reset_index())

# Merge Baseline and Promoted Store Category Averages
df_store_lift = baseline_store_comm.merge(
    promoted_store_comm, on=["store_id", "commodity_desc"], how="inner")

print(
    f"Store-Commodity Pairs with both Baseline and Promoted Weeks:"
    f" {len(df_store_lift)}"
    )
df_store_lift.head()

Store-Commodity Pairs with both Baseline and Promoted Weeks: 3042


,store_id,commodity_desc,baseline_weekly_sales,baseline_weeks_count,promoted_weekly_sales,promoted_weeks_count
0,289,FROZEN PIZZA,2.000000,1,2.500000,1
1,289,FRZN MEAT/MEAT DINNERS,1.875000,4,1.000000,1
2,292,APPLES,1.590000,1,2.000000,1
3,292,BAG SNACKS,2.054286,6,2.546667,3
4,292,BAKED BREAD/BUNS/ROLLS,1.590000,6,1.790000,3


In [ ]:
# compute weekly store promotional Lift & clean catch-all categories
df_store_lift["weekly_sales_lift"] = (
    df_store_lift["promoted_weekly_sales"] - df_store_lift["baseline_weekly_sales"])

# Calculate Percentage Lift
df_store_lift["lift_percentage"] = (
    (df_store_lift["weekly_sales_lift"] / df_store_lift["baseline_weekly_sales"]) * 100)

# Add Catch-all Flag for accounting/non-commodity/unknown items
df_store_lift["is_catchall_category"] = df_store_lift["commodity_desc"].isin(
    ["COUPON/MISC ITEMS", "NO COMMODITY DESCRIPTION", "UNKNOWN", ""])

# Filter for clean commodities and display top store-level promotional lifts
df_clean_store_lift = df_store_lift[df_store_lift["is_catchall_category"] == False]

print("--- Top 5 Stores & Categories with Highest Weekly Promotional Lift ---")
print(
    df_clean_store_lift.sort_values("weekly_sales_lift", ascending=False)[
        [
            "store_id",
            "commodity_desc",
            "baseline_weekly_sales",
            "promoted_weekly_sales",
            "weekly_sales_lift",
            "lift_percentage",
        ]
    ].head(5).to_string(index=False)
)

--- Top 5 Stores & Categories with Highest Weekly Promotional Lift ---
 store_id               commodity_desc  baseline_weekly_sales  promoted_weekly_sales  weekly_sales_lift  lift_percentage
      415 PREPAID WIRELESS&ACCESSORIES                  10.00                  39.99              29.99       299.900000
      367          MEAT - SHELF STABLE                   3.00                  27.90              24.90       830.000000
      391                 SMOKED MEATS                  14.15                  35.55              21.40       151.236749
      415          CANDLES/ACCESSORIES                   4.37                  22.73              18.36       420.137300
      406               COFFEE FILTERS                  22.99                  39.99              17.00        73.945194


In [20]:
# Summary:
# - Promotion coverage: 14,526 out of 69,145 store-product weeks (21%) had active promotions.
# - Highest promotional lift observed:
#   - Meat - Shelf Stable → +830% lift
#   - Candles / Accessories → +420.1% lift
#   - Prepaid Wireless & Accessories → +299.9% lift
# - Linking silver.causal_data with sales data enables effective measurement of merchandising impact.

In [ ]:
# Load all sample household daily discounts & sales
df_hh_daily = (
    df_trans_sample.groupby(["household_key", "day", "trans_date"]).agg(
        actual_sales=("sales_value", "sum"),
        retail_disc=("retail_disc", lambda x: sum(abs(x))),
        coupon_disc=("coupon_disc", lambda x: sum(abs(x))),
        coupon_match_disc=("coupon_match_disc", lambda x: sum(abs(x))),).reset_index())

# Total discounts given
df_hh_daily["total_discounts"] = (
    df_hh_daily["retail_disc"]
    + df_hh_daily["coupon_disc"]
    + df_hh_daily["coupon_match_disc"])

print(f"Total Household-Daily Records: {len(df_hh_daily):,}")
df_hh_daily.head()

Total Household-Daily Records: 6,407


,household_key,day,trans_date,actual_sales,retail_disc,coupon_disc,coupon_match_disc,total_discounts
0,1,51,2020-02-20,78.66,16.54,1.0,0.0,17.54
1,1,67,2020-03-07,41.10,8.59,0.0,0.0,8.59
2,1,88,2020-03-28,26.90,6.72,0.0,0.0,6.72
3,1,94,2020-04-03,63.43,11.08,1.5,0.5,13.08
4,1,101,2020-04-10,53.45,16.42,0.0,0.0,16.42


In [ ]:
# attach campaign exposure & compute household overall baseline daily rate
# 1. Attach campaign exposure flag
df_hh_daily["is_during_campaign"] = df_hh_daily.apply(
    lambda row: is_campaign_active(row["household_key"], row["day"]), axis=1)

# 2. Compute overall non-campaign baseline per household
hh_daily_baselines = (
    df_hh_daily[df_hh_daily["is_during_campaign"] == False].groupby("household_key").agg(
        baseline_total_sales=("actual_sales", "sum"),
        baseline_active_days=("day", "nunique"),).reset_index())

hh_daily_baselines["household_daily_baseline_rate"] = (
    hh_daily_baselines["baseline_total_sales"]
    / hh_daily_baselines["baseline_active_days"])

# 3. Merge baseline rate back into daily records
df_hh_daily = df_hh_daily.merge(
    hh_daily_baselines[["household_key", "household_daily_baseline_rate"]], on="household_key", how="left",).fillna({"household_daily_baseline_rate": 0})

print("Daily Records with Household Baseline Rate:")
df_hh_daily[["household_key", "day", "actual_sales", "household_daily_baseline_rate", "is_during_campaign"]].head()

Daily Records with Household Baseline Rate:


,household_key,day,actual_sales,household_daily_baseline_rate,is_during_campaign
0,1,51,78.66,62.14122,False
1,1,67,41.10,62.14122,False
2,1,88,26.90,62.14122,False
3,1,94,63.43,62.14122,False
4,1,101,53.45,62.14122,False


In [23]:
# compute daily incremental sales lift & apply floored wasted spend logic
# 1. Calculate Daily Incremental Lift
df_hh_daily["daily_incremental_lift"] = (
    df_hh_daily["actual_sales"] - df_hh_daily["household_daily_baseline_rate"])

# 2. Apply Floored Wasted Spend Rule
df_hh_daily["wasted_spend_floored"] = df_hh_daily.apply(
    lambda row: row["total_discounts"]
    if (row["is_during_campaign"] and row["daily_incremental_lift"] <= 0 and row["total_discounts"] > 0)
    else 0.0,
    axis=1,)

print(f"Total Discounts Given in Sample: ${df_hh_daily['total_discounts'].sum():,.2f}")
print(f"Total Floored Wasted Spend: ${df_hh_daily['wasted_spend_floored'].sum():,.2f}")

Total Discounts Given in Sample: $38,719.48
Total Floored Wasted Spend: $3,497.67


In [ ]:
# roll up to daily executive gold metrics summary
# Roll up to executive daily view
df_exec_daily = (
    df_hh_daily.groupby(["day", "trans_date"]).agg(
        actual_sales=("actual_sales", "sum"),
        expected_baseline_sales=("household_daily_baseline_rate", "sum"),
        total_discounts=("total_discounts", "sum"),
        wasted_spend_floored=("wasted_spend_floored", "sum"),).reset_index())

# Calculate Daily Wasted Spend Percentage
df_exec_daily["wasted_spend_pct"] = np.where(
    df_exec_daily["total_discounts"] > 0, (df_exec_daily["wasted_spend_floored"] / df_exec_daily["total_discounts"]) * 100, 0.0,)

print(f"Executive Days Summarized: {len(df_exec_daily)}")
df_exec_daily.head(10)

Executive Days Summarized: 695


,day,trans_date,actual_sales,expected_baseline_sales,total_discounts,wasted_spend_floored,wasted_spend_pct
0,4,2020-01-04,68.69,28.318833,19.75,0.0,0.0
1,5,2020-01-05,8.13,12.211634,1.76,0.0,0.0
2,10,2020-01-10,400.79,19.327986,0.00,0.0,0.0
3,11,2020-01-11,141.59,61.560285,31.11,0.0,0.0
4,12,2020-01-12,1.55,12.211634,0.29,0.0,0.0
5,13,2020-01-13,35.43,39.944800,2.64,0.0,0.0
6,14,2020-01-14,145.81,105.766279,19.06,0.0,0.0
7,15,2020-01-15,27.45,34.814643,3.23,0.0,0.0
8,18,2020-01-18,13.71,47.026277,1.32,0.0,0.0
9,19,2020-01-19,2.00,12.211634,1.87,0.0,0.0


In [25]:
# print phase complete executive metrics summary
total_actual = df_exec_daily["actual_sales"].sum()
total_baseline = df_exec_daily["expected_baseline_sales"].sum()
total_disc = df_exec_daily["total_discounts"].sum()
total_wasted = df_exec_daily["wasted_spend_floored"].sum()
overall_wasted_pct = (total_wasted / total_disc) * 100 if total_disc > 0 else 0

print("==================================================")
print("       EXECUTIVE SUMMARY METRICS (PROTOTYPE)      ")
print("==================================================")
print(f"Total Actual Sales:           ${total_actual:,.2f}")
print(f"Expected Baseline Sales:      ${total_baseline:,.2f}")
print(f"Total Discounts Provided:     ${total_disc:,.2f}")
print(f"Floored Wasted Discount Spend:${total_wasted:,.2f}")
print(f"Overall Wasted Spend %:       {overall_wasted_pct:.2f}%")
print("==================================================")

       EXECUTIVE SUMMARY METRICS (PROTOTYPE)      
Total Actual Sales:           $225,473.63
Expected Baseline Sales:      $224,126.89
Total Discounts Provided:     $38,719.48
Floored Wasted Discount Spend:$3,497.67
Overall Wasted Spend %:       9.03%
